# Final Model — LogisticRegression + Custom Features

The chosen model, trained once with fixed hyperparameters. **No Optuna here.**

Those hyperparameters came from two earlier phases:
1. **Algorithm comparison** (`experiment_5_*`) — LogReg beat SVM, LightGBM, KNN, RF, NB, XGBoost
2. **Feature engineering** (`custom_features.ipynb`) — 50 Optuna trials over ~10 hours

This notebook does four things:
- retrains the winner and confirms it reproduces the reported scores
- logs it to MLflow
- **saves a complete artifact** (vectorizer + column order + model) in one file
- provides `predict_comments()` so raw text can be scored directly

That last part matters: a saved classifier on its own is useless, because it needs the
exact TF-IDF vocabulary it was trained with.

In [1]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MaxAbsScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack, csr_matrix
import spacy
import mlflow
import mlflow.sklearn

# MLflow / DagShub tracking setup
os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"
mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")
mlflow.set_experiment("Final Model - LogReg + Custom Features")

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/07/22 17:32:24 INFO mlflow.tracking.fluent: Experiment with name 'Final Model - LogReg + Custom Features' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/dbcd559c999a41cbac7e8c53f1b36f97', creation_time=1784719945420, effective_trace_archival_retention=None, experiment_id='15', last_update_time=1784719945420, lifecycle_stage='active', name='Final Model - LogReg + Custom Features', tags={}, trace_location=None, workspace='default'>

## The chosen configuration

Everything that defines the model lives in one place, so there is no hunting through cells.

In [2]:
# Winner of custom_features.ipynb (50 Optuna trials, ~500 minutes)
# Reported there: accuracy 0.8871 | macro F1 0.8756
BEST_PARAMS = {
    'C':            1.1199134843600127,
    'l1_ratio':     0.9990014673384794,   # ~1.0 => essentially pure L1
    'class_weight': None,
    'solver':       'saga',
    'max_iter':     300,
    'random_state': 42,
}

NGRAM_RANGE  = (1, 2)
MAX_FEATURES = 10000

BEST_PARAMS

{'C': 1.1199134843600127,
 'l1_ratio': 0.9990014673384794,
 'class_weight': None,
 'solver': 'saga',
 'max_iter': 300,
 'random_state': 42}

In [3]:
# Load dataset
dataset = pd.read_csv('dataset.csv')
cleaned_dataset = dataset.dropna()

print('rows:', len(cleaned_dataset))
cleaned_dataset['category'].value_counts()

rows: 36662


category
 1    15770
 0    12644
-1     8248
Name: count, dtype: int64

In [4]:
X_cleaned = cleaned_dataset['clean_comment']
y_cleaned = cleaned_dataset['category']

# Same split as custom_features.ipynb (test_size=0.2, random_state=42, no stratify)
# so the scores below are directly comparable to the 0.8871 reported there.
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42
)

print('train:', len(X_train_text), ' test:', len(X_test_text))

train: 29329  test: 7333


In [5]:
# Load spacy language model for POS tagging
nlp = spacy.load('en_core_web_sm')

In [6]:
# All 17 universal POS tags, in a FIXED order.
# Looping over this (instead of set(pos_tags)) guarantees every comment produces the
# same 17 POS columns in the same order -> train, test, and future data always line up.
UNIVERSAL_POS = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM',
                 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

def extract_custom_features(text):
    """Turn one raw comment into 6 statistics + 17 POS proportions."""
    doc = nlp(text)
    word_list = [token.text for token in doc]

    comment_length    = len(text)
    word_count        = len(word_list)
    avg_word_length   = sum(len(w) for w in word_list) / word_count if word_count > 0 else 0
    unique_word_count = len(set(word_list))
    lexical_diversity = unique_word_count / word_count if word_count > 0 else 0
    pos_count         = len([token.pos_ for token in doc])

    pos_tags = [token.pos_ for token in doc]
    if word_count > 0:
        pos_proportion = {tag: pos_tags.count(tag) / word_count for tag in UNIVERSAL_POS}
    else:
        pos_proportion = {tag: 0 for tag in UNIVERSAL_POS}

    return {
        'comment_length': comment_length,
        'word_count': word_count,
        'avg_word_length': avg_word_length,
        'unique_word_count': unique_word_count,
        'lexical_diversity': lexical_diversity,
        'pos_count': pos_count,
        **pos_proportion
    }

## Feature extraction

⏳ The spaCy pass is the slow step — a few minutes over ~36k comments.

In [7]:
train_custom_features = pd.DataFrame([extract_custom_features(t) for t in X_train_text])
test_custom_features  = pd.DataFrame([extract_custom_features(t) for t in X_test_text])

# Lock the column order. This exact list must be reproduced at inference time,
# which is why it gets saved alongside the model further down.
CUSTOM_COLUMNS = list(train_custom_features.columns)
test_custom_features = test_custom_features.reindex(columns=CUSTOM_COLUMNS, fill_value=0)

print('custom feature columns:', len(CUSTOM_COLUMNS))
print(CUSTOM_COLUMNS)

custom feature columns: 23
['comment_length', 'word_count', 'avg_word_length', 'unique_word_count', 'lexical_diversity', 'pos_count', 'ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']


In [8]:
# TF-IDF - fit ONLY on training text
tfidf = TfidfVectorizer(ngram_range=NGRAM_RANGE, max_features=MAX_FEATURES)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf  = tfidf.transform(X_test_text)

print('tfidf vocabulary size:', len(tfidf.vocabulary_))

tfidf vocabulary size: 10000


In [9]:
# Combine [sparse TF-IDF | custom features] while STAYING SPARSE.
# Densifying here would cost ~2.3 GB and make training unusably slow.
X_train = hstack([X_train_tfidf, csr_matrix(train_custom_features[CUSTOM_COLUMNS].values)]).tocsr()
X_test  = hstack([X_test_tfidf,  csr_matrix(test_custom_features[CUSTOM_COLUMNS].values)]).tocsr()

print('train:', X_train.shape)
print('test :', X_test.shape)
print(f'density: {X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.2f}% non-zero')

train: (29329, 10023)
test : (7333, 10023)
density: 0.27% non-zero


## Train

One fit, fixed hyperparameters. MaxAbsScaler is used because it is sparse-safe —
StandardScaler would centre the data and destroy the sparsity.

In [10]:
model = Pipeline([
    ('scaler', MaxAbsScaler()),
    ('clf', LogisticRegression(**BEST_PARAMS)),
])

model.fit(X_train, y_train)
print('training complete')

training complete


C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [11]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report   = classification_report(y_test, y_pred, output_dict=True)

print(f'Accuracy : {accuracy:.4f}    (custom_features.ipynb reported 0.8871)')
print(f'Macro F1 : {report["macro avg"]["f1-score"]:.4f}    (reported 0.8756)')
print()
print(classification_report(y_test, y_pred))

Accuracy : 0.8871    (custom_features.ipynb reported 0.8871)
Macro F1 : 0.8756    (reported 0.8756)

              precision    recall  f1-score   support

          -1       0.87      0.76      0.81      1647
           0       0.87      0.97      0.92      2510
           1       0.91      0.89      0.90      3176

    accuracy                           0.89      7333
   macro avg       0.88      0.87      0.88      7333
weighted avg       0.89      0.89      0.89      7333



## Log to MLflow

In [12]:
with mlflow.start_run():
    mlflow.set_tag("mlflow.runName", "FinalModel_LogReg_CustomFeatures")
    mlflow.set_tag("stage", "final_model")

    mlflow.log_param("vectorizer_type", "TfidfVectorizer")
    mlflow.log_param("ngram_range", str(NGRAM_RANGE))
    mlflow.log_param("vectorizer_max_features", MAX_FEATURES)
    mlflow.log_param("scaler", "MaxAbsScaler")
    mlflow.log_param("n_custom_features", len(CUSTOM_COLUMNS))
    for k, v in BEST_PARAMS.items():
        mlflow.log_param(k, v)

    mlflow.log_metric("accuracy", accuracy)
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            for m, v in metrics.items():
                mlflow.log_metric(f"{label}_{m}", v)

    # Save locally FIRST so a logging failure can never lose the trained model
    joblib.dump(model, 'model_b_classifier.joblib')
    try:
        mlflow.sklearn.log_model(
            model, "model_b",
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
        )
    except Exception as e:
        print(f"[warn] MLflow model logging failed (metrics + joblib file are safe): {e}")

print('logged to MLflow')

2026/07/22 18:01:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/22 18:01:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run FinalModel_LogReg_CustomFeatures at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/15/runs/6aaf6181c47e4d6a9a463240a4151955
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/15
logged to MLflow


## Save a complete, reusable artifact

A trained classifier on its own **cannot** score a new comment — it needs the exact
TF-IDF vocabulary it was trained on, plus the exact custom-feature column order.

So all three pieces are saved together in one file.

In [13]:
artifact = {
    'tfidf':          tfidf,            # fitted vectorizer (vocabulary + IDF weights)
    'model':          model,            # Pipeline(MaxAbsScaler -> LogisticRegression)
    'custom_columns': CUSTOM_COLUMNS,   # exact column order the model expects
    'universal_pos':  UNIVERSAL_POS,
    'params':         BEST_PARAMS,
    'spacy_model':    'en_core_web_sm',
    'ngram_range':    NGRAM_RANGE,
    'max_features':   MAX_FEATURES,
}

joblib.dump(artifact, 'model_b_final.joblib')
print('saved -> model_b_final.joblib')

saved -> model_b_final.joblib


## Scoring raw comments

This is what the evaluation dataset will use.

⚠️ Note: `predict_comments` depends on `extract_custom_features`, which needs `nlp`
loaded in the session. For the production pipeline this function should move into
`src/` so it can be imported anywhere rather than living in a notebook.

In [19]:
def predict_comments(comments, artifact):
    """Raw comment text -> predicted labels (-1 negative, 0 neutral, 1 positive)."""
    feats = pd.DataFrame([extract_custom_features(c) for c in comments])
    feats = feats.reindex(columns=artifact['custom_columns'], fill_value=0)

    X = hstack([
        artifact['tfidf'].transform(comments),
        csr_matrix(feats.values),
    ]).tocsr()

    return artifact['model'].predict(X)


samples = [
    "this video was absolutely terrible and a complete waste of time",
    "nothing",
    "amazing work, i loved every second of this",
]

for comment, pred in zip(samples, predict_comments(samples, artifact)):
    print(f'{pred:>3}  |  {comment}')

 -1  |  this video was absolutely terrible and a complete waste of time
  0  |  nothing
  1  |  amazing work, i loved every second of this
